<img align="right" src="https://panoptes-uploads.zooniverse.org/project_avatar/86c23ca7-bbaa-4e84-8d8a-876819551431.png" type="image/png" height=100 width=100>
</img>

# Train and Evaluate Models

**KSO2 Notebook 2 of 5** - Written by the KSO Team

---

Train YOLO detection models on your project's dataset and compare their validation results to choose a model. Every training run is logged to MLflow, so you can try different model sizes and settings without filling the project YAML with experiments.

Once you have chosen a model, test it once on the test split for the final performance estimate. The test split is kept apart until then, so this score is honest. The chosen model, its training settings and its test results are then recorded in the project YAML, for the next notebooks.

### Prerequisites

- **Notebook 1: Project Setup**, run to the end. You need the `YAML_PATH` it printed.
- A session with a GPU. Training on a CPU works but is much slower.

**Steps:** load the project, train and compare candidate models, then test and record the model you choose.

**Next:** Notebook 3: Model Inference.

---
## Phase 0: Setup

Imports KSO2. The output shows which copy of KSO is running and which GPU the session has.

In [ ]:
import io
import logging
import os
from contextlib import redirect_stdout
from pathlib import Path
from urllib.parse import urlparse

import mlflow
import torch
import yaml
from ultralytics import YOLO

import kso
from kso import ProjectManager, TrainingManager

logging.getLogger("ultralytics").disabled = False  # importing kso turns it off

proj = ProjectManager()
trainer = TrainingManager()

gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None

print("[OK] KSO2 imported")
print(f"  running from: {kso.__file__}")
print(f"  GPU: {gpu or 'none, training runs on the CPU (much slower)'}")

---
## Phase 1: Load the Project

Paste the `YAML_PATH` printed by Notebook 1, then run the cell.

In [ ]:
YAML_PATH = "/path/to/projects/my_project_name/my_project_name.project.yaml"  # <-- EDIT

with redirect_stdout(io.StringIO()):  # skips the project YAML printout
    project = proj.load_project(yaml_path=YAML_PATH)
project_dir = Path(project.Config_file_path).resolve().parent
data_yaml = project.data_path["ultralytics_data_path"]
mlflow.set_tracking_uri(f"sqlite:///{project.tracking}")  # the project's MLflow log

records = proj.yaml_data_retrieve(yaml_path=project.Config_file_path)
datasets = [d for d in records.get("datasets") or [] if d.get("data_yaml") == data_yaml]
if not datasets:
    raise RuntimeError(f"No dataset recorded for {data_yaml}. Run Notebook 1 first.")
dataset = datasets[0]
if dataset["task"] != "detect":
    raise RuntimeError(f"This notebook trains detection models, not {dataset['task']}.")

print(f"\n[OK] Project loaded: {project.project_name}")
print(f"  Dataset: {dataset['name']}, {len(dataset['class_names'])} classes")
for split in ["train", "val", "test"]:
    print(f"  {split:5s}: {dataset['split'][split + '_images']:5d} images")

---
## Phase 2: Train a Model

Edit the values and run the cell. To train more models to compare, run it again with other values.

- `MODEL_PATH`: the model to start from. Pick one from the table below, or give the full path to your own `.pt` file.
- `EPOCHS`: how many passes over the training images. 50 is a good start.
- `MODEL_PREFIX`: a short label for your data, such as `baltic`.
- `BATCH_SIZE`: lower it if training runs out of GPU memory.

The model is named from `MODEL_PREFIX`, the model file and `EPOCHS`, for example `baltic_yolo26s_50ep` Training again under the same name replaces that model.

| Family | Models, smallest to largest |
|---|---|
| YOLO11 | `yolo11n.pt`, `yolo11s.pt`, `yolo11m.pt`, `yolo11l.pt`, `yolo11x.pt` |
| YOLO26 | `yolo26n.pt`, `yolo26s.pt`, `yolo26m.pt`, `yolo26l.pt`, `yolo26x.pt` |

Start with a small model (`n` or `s`). Move to a larger one only if you have the computing time and it does better in Phase 3. As a rough guide: `n` or `s` below about 250 training images, `m` up to about 800, and `l` or `x` above that.

During training, each epoch shows the training losses and the validation results (P, R, mAP50, mAP50-95). Look for losses generally decreasing and validation performance improving or stabilising over training.

In [ ]:
MODEL_PATH = "yolo11n.pt"  # <-- EDIT
EPOCHS = 50  # <-- EDIT
MODEL_PREFIX = "my_data"  # <-- EDIT, for example "baltic"
BATCH_SIZE = 8
IMG_SIZE = 640

MODEL_NAME = f"{MODEL_PREFIX}_{Path(MODEL_PATH).stem}_{EPOCHS}ep"

with redirect_stdout(io.StringIO()):  # skips the project YAML printout
    proj.add_model(project=project, model_path=MODEL_PATH, model_name=MODEL_NAME)
results = trainer.train_model(
    project=project, epochs=EPOCHS, imgsz=IMG_SIZE, batch=BATCH_SIZE
)

print(f"\n[OK] Trained '{project.model_name}'")
print(f"  Run folder: {results.save_dir}")

---
## Phase 3: Compare Your Models

Lists every model trained in this project with its validation results, best `mAP50` first. All scores run from 0 to 1, and higher is better.

- `precision`: the share of detections that are correct. Low precision means false alarms.
- `recall`: the share of objects that are found of all real objects present in the images. High recall = few misses. A recall of 0.80 means the model catches 80% of objects. 
- `mAP`: Mean Average Precision, a single summary score of detection quality across confidence thresholds. Higher is better; reported as `mAP@50` (lenient) and `mAP@50-95` (strict).

Don't choose on `mAP50` alone: it can look good while precision or recall is poor. Before choosing, open the model's folder, `runs/detect/<model_name>` in your project folder, and look at the different graphs and curves. The curves should improve and then level off. Validation losses that rise again towards the end mean the model is overfitting: try fewer epochs or a smaller model. `confusion_matrix.png` in the same folder shows which classes the model mixes up. See https://docs.ultralytics.com/guides/yolo-performance-metrics for more details.

In [ ]:
runs = mlflow.search_runs(
    experiment_names=[project.project_name],
    filter_string="attributes.status = 'FINISHED'",
    order_by=["attributes.start_time DESC"],
)
if runs.empty:
    raise RuntimeError("No trained models yet. Run Phase 2 first.")
runs["model"] = runs["tags.mlflow.runName"].str.rsplit("-", n=1).str[0]
runs = runs.drop_duplicates("model")  # keeps the latest training of each model

columns = {
    "model": "model",
    "metrics.metrics/precision": "precision",
    "metrics.metrics/recall": "recall",
    "metrics.metrics/mAP50": "mAP50",
    "metrics.metrics/mAP50-95": "mAP50-95",
}
table = runs[list(columns)].rename(columns=columns)
print("Validation results:")
print(table.sort_values("mAP50", ascending=False).round(3).to_string(index=False))

---
## Phase 4: Test and Record the Chosen Model

The test split was kept apart during training and comparison, so it gives an honest final score. Use it once, for the model you choose.

Set `FINAL_MODEL` to a name from the table and run the cell. It tests that model and records it in the project YAML, with its training settings and test results. While `FINAL_MODEL` is empty, nothing is tested or recorded. In a later session, run Phases 0, 1 and 3, then this cell.

The test keeps predictions down to a confidence of 0.001 (`conf_floor`), so that mAP covers the whole precision-recall curve. This is not the detection threshold, which you choose in Notebook 3. The per-class table is for checking only: a class with few test instances gives unstable numbers, so only the overall results are recorded. A class missing from that table had no test images.

In [ ]:
FINAL_MODEL = ""  # <-- EDIT once chosen, for example "baltic_yolo26m_50ep"

if not FINAL_MODEL:
    print("[SKIP] No model chosen yet, so the test split was not used.")
else:
    chosen = runs[runs["model"] == proj.sanitized_name(FINAL_MODEL)]
    if chosen.empty:
        raise RuntimeError(f"'{FINAL_MODEL}' is not in the table of Phase 3.")
    run = chosen.iloc[0]
    args_file = Path(urlparse(run["artifact_uri"]).path) / "yolo" / "args.yaml"
    args = yaml.safe_load(args_file.read_text(encoding="utf-8"))
    if args["data"] != data_yaml:
        raise RuntimeError(f"'{FINAL_MODEL}' was trained on another dataset.")
    run_dir = Path(args["save_dir"])
    weights = run_dir / "weights" / "best.pt"
    metrics = YOLO(str(weights)).val(
        data=data_yaml,
        split="test",
        conf=0.001,
        imgsz=args["imgsz"],
        batch=args["batch"],
        project=str(run_dir),
        name="test_eval",
        exist_ok=True,
    )
    test = {
        "map50": round(float(metrics.box.map50), 4),
        "map50_95": round(float(metrics.box.map), 4),
        "precision": round(float(metrics.box.mp), 4),
        "recall": round(float(metrics.box.mr), 4),
    }

    entry = {
        "model_name": run["model"],
        "model_path": args["model"],
        "task": dataset["task"],
        "dataset": dataset["name"],
        "run_dir": os.path.relpath(run_dir, project_dir),
        "weights": os.path.relpath(weights, project_dir),
        "mlflow": {"experiment": project.project_name, "run_id": run["run_id"]},
        "training": {key: args[key] for key in ["epochs", "imgsz", "batch"]},
        "evaluation": {"split": "test", "conf_floor": 0.001, **test},
    }
    proj.record(project, "models", entry)

    print(f"\n[OK] Recorded '{entry['model_name']}' with its test results")
    for key, value in test.items():
        print(f"  test {key:9s} {value:.3f}")

---
## Done

Your project folder now holds:

```
<project_name>/
|-- <project_name>.project.yaml   the dataset and the models, the chosen one with its test results
|-- mlflow.db                     every training run, compared in Phase 3
|-- mlruns/                       MLflow files
`-- runs/detect/<model_name>/     one folder per training
    |-- weights/best.pt           the trained model
    |-- results.png               training curves, to check before choosing
    `-- test_eval/                test plots, for the chosen model
```

Report the test results of the model you chose. Continue to **Notebook 3: Model Inference** with the same `YAML_PATH` and your `FINAL_MODEL`.